# EDA — ADFA-LD Dataset

**Objectif:** Comprendre les données AVANT de modéliser.

**Phase 1 du plan** — voir `PLAN.md` et `EXPLICATION_DATA.md`.

Ce notebook répond aux questions :
1. Combien de fichiers a-t-on, par classe et par famille ?
2. Comment sont distribuées les longueurs des séquences ?
3. Quels syscalls dominent dans chaque classe ?
4. Y a-t-il des fichiers anormaux (vides, malformés, noms bizarres) ?
5. Quelles règles de nettoyage appliquer ?

## 0. Setup

In [ ]:
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# Chemins relatifs depuis notebooks/
BASE = Path('../data/ADFA-LD')
TRAINING = BASE / 'Training_Data_Master'
VALIDATION = BASE / 'Validation_Data_Master'
ATTACK = BASE / 'Attack_Data_Master'

# Dossier de sortie pour les figures et rapports
OUT = Path('../results/eda')
OUT.mkdir(parents=True, exist_ok=True)

print(f'Training_Data_Master existe : {TRAINING.exists()}')
print(f'Validation_Data_Master existe : {VALIDATION.exists()}')
print(f'Attack_Data_Master existe : {ATTACK.exists()}')

## 1. Chargement des données

On parcourt les 3 dossiers et on construit un DataFrame avec une ligne par fichier.

In [ ]:
def load_syscall_file(path):
    """Lit un fichier de trace et retourne (sequence_str, length, valid).

    - sequence_str : contenu nettoyé (espaces normalisés)
    - length : nombre de tokens (syscalls)
    - valid : True si tous les tokens sont des entiers positifs
    """
    try:
        text = Path(path).read_text(encoding='utf-8', errors='ignore').strip()
        if not text:
            return '', 0, True
        tokens = text.split()
        valid = all(t.isdigit() for t in tokens)
        return ' '.join(tokens), len(tokens), valid
    except Exception as e:
        print(f'Erreur lecture {path}: {e}')
        return '', 0, False

In [ ]:
rows = []

# --- Training_Data_Master (NORMAL, label=0) ---
for path in sorted(TRAINING.glob('*.txt')):
    seq, length, valid = load_syscall_file(path)
    rows.append({
        'filename': path.name,
        'source': 'Training',
        'label': 0,
        'family': 'normal',
        'scenario': f'normal_{path.stem}',  # chaque fichier normal est son propre groupe
        'sequence': seq,
        'length': length,
        'valid': valid
    })

# --- Validation_Data_Master (NORMAL, label=0) ---
for path in sorted(VALIDATION.glob('*.txt')):
    seq, length, valid = load_syscall_file(path)
    rows.append({
        'filename': path.name,
        'source': 'Validation',
        'label': 0,
        'family': 'normal',
        'scenario': f'normal_{path.stem}',
        'sequence': seq,
        'length': length,
        'valid': valid
    })

# --- Attack_Data_Master (ATTAQUE, label=1) ---
for scenario_dir in sorted(ATTACK.iterdir()):
    if not scenario_dir.is_dir():
        continue
    scenario_name = scenario_dir.name              # ex: 'Adduser_1'
    family = re.sub(r'_\d+$', '', scenario_name)   # ex: 'Adduser'
    for path in sorted(scenario_dir.glob('*.txt')):
        seq, length, valid = load_syscall_file(path)
        rows.append({
            'filename': path.name,
            'source': 'Attack',
            'label': 1,
            'family': family,
            'scenario': scenario_name,
            'sequence': seq,
            'length': length,
            'valid': valid
        })

df = pd.DataFrame(rows)
print(f'TOTAL fichiers chargés : {len(df)}')
df.head(3)

## 2. Statistiques de base

In [ ]:
print('=== Répartition par classe (label) ===')
print(df['label'].value_counts().rename({0: 'Normal', 1: 'Attaque'}))
print()
print('=== Répartition par source ===')
print(df['source'].value_counts())
print()
print('=== Répartition par famille ===')
print(df['family'].value_counts())
print()
print(f'Nb scénarios d\'attaque distincts : {df[df.label==1]["scenario"].nunique()}')

## 3. Distribution des longueurs

In [ ]:
print(df.groupby('label')['length'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df[df.label==0]['length'].hist(bins=60, ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Normal — distribution des longueurs')
axes[0].set_xlabel('Nombre de syscalls par fichier')
axes[0].set_ylabel('Nombre de fichiers')

df[df.label==1]['length'].hist(bins=60, ax=axes[1], color='crimson', alpha=0.8)
axes[1].set_title('Attaque — distribution des longueurs')
axes[1].set_xlabel('Nombre de syscalls par fichier')

plt.tight_layout()
plt.savefig(OUT / 'length_distribution.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
order = ['normal', 'Adduser', 'Hydra_FTP', 'Hydra_SSH', 'Java_Meterpreter', 'Meterpreter', 'Web_Shell']
sns.boxplot(data=df, x='family', y='length', order=order, palette='Set2')
plt.title('Longueurs par famille')
plt.xticks(rotation=20)
plt.ylim(0, 3000)
plt.savefig(OUT / 'length_by_family.png', dpi=110, bbox_inches='tight')
plt.show()

## 4. Top syscalls par classe

In [ ]:
def count_syscalls(df_subset):
    counter = Counter()
    for seq in df_subset['sequence']:
        if seq:
            counter.update(seq.split())
    return counter

normal_counts = count_syscalls(df[df.label == 0])
attack_counts = count_syscalls(df[df.label == 1])

top_normal = normal_counts.most_common(20)
top_attack = attack_counts.most_common(20)

print('Top 20 syscalls — NORMAL')
for s, c in top_normal:
    print(f'  syscall {s:>4} : {c:>8} appels')

print('\nTop 20 syscalls — ATTAQUE')
for s, c in top_attack:
    print(f'  syscall {s:>4} : {c:>8} appels')

In [ ]:
# Comparaison visuelle : fréquences RELATIVES (% des syscalls totaux)
common = list({s for s, _ in top_normal[:15]} | {s for s, _ in top_attack[:15]})

tot_n = sum(normal_counts.values())
tot_a = sum(attack_counts.values())

rows = []
for s in common:
    rows.append({'syscall': s, 'classe': 'Normal',  'freq_rel': normal_counts.get(s, 0) / tot_n})
    rows.append({'syscall': s, 'classe': 'Attaque', 'freq_rel': attack_counts.get(s, 0) / tot_a})

freq_df = pd.DataFrame(rows).sort_values('freq_rel', ascending=False)

plt.figure(figsize=(14, 5))
sns.barplot(data=freq_df, x='syscall', y='freq_rel', hue='classe', palette={'Normal': 'steelblue', 'Attaque': 'crimson'})
plt.title('Fréquence relative des syscalls (top 15 cumulés)')
plt.ylabel('Fréquence relative (% des syscalls totaux de la classe)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUT / 'top_syscalls_comparison.png', dpi=110, bbox_inches='tight')
plt.show()

## 5. Détection d'anomalies

In [ ]:
print('=== Fichiers vides (length == 0) ===')
empty = df[df.length == 0]
print(f'  Total : {len(empty)}')
if len(empty) > 0:
    print(empty[['filename', 'family', 'scenario']].head(10))

print('\n=== Fichiers très courts (0 < length < 10) ===')
short = df[(df.length > 0) & (df.length < 10)]
print(f'  Total : {len(short)}')
if len(short) > 0:
    print(short[['filename', 'family', 'scenario', 'length']].head(10))

print('\n=== Fichiers invalides (tokens non-entiers) ===')
invalid = df[~df.valid]
print(f'  Total : {len(invalid)}')
if len(invalid) > 0:
    print(invalid[['filename', 'family', 'scenario', 'length']].head(10))

print('\n=== Noms de fichier étranges (contiennent "=") ===')
strange = df[df.filename.str.contains('=')]
print(f'  Total : {len(strange)}')
if len(strange) > 0:
    print(strange[['filename', 'family', 'scenario', 'length']])

## 6. Équilibre des classes

In [ ]:
counts = df['label'].value_counts()
n_normal = int(counts.get(0, 0))
n_attack = int(counts.get(1, 0))
ratio = n_normal / max(n_attack, 1)

print(f'Normal  : {n_normal}')
print(f'Attaque : {n_attack}')
print(f'Ratio normal:attaque = {ratio:.2f} : 1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].map({0: 'Normal', 1: 'Attaque'}).value_counts().plot.pie(
    autopct='%.1f%%', colors=['steelblue', 'crimson'], ax=axes[0]
)
axes[0].set_title('Distribution des classes')
axes[0].set_ylabel('')

df['family'].value_counts().plot.bar(ax=axes[1], color='slategray')
axes[1].set_title('Fichiers par famille')
axes[1].set_xlabel('Famille')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(OUT / 'class_balance.png', dpi=110, bbox_inches='tight')
plt.show()

## 7. Application des règles de nettoyage

Règles décidées dans `PLAN.md` :
1. Rejeter `length == 0` (vides)
2. Rejeter `length < 10` (trop courts pour des trigrammes utiles)
3. Rejeter les fichiers invalides (tokens non-entiers)

In [ ]:
df['keep'] = (df.length >= 10) & df.valid

kept = int(df['keep'].sum())
rejected = int((~df['keep']).sum())

print(f'Avant nettoyage : {len(df)} fichiers')
print(f'  À garder      : {kept}')
print(f'  À rejeter     : {rejected}')

print('\nDétail des rejets :')
rej = df[~df.keep]
print(f'  Vides (length == 0)        : {(rej.length == 0).sum()}')
print(f'  Trop courts (< 10)         : {((rej.length > 0) & (rej.length < 10)).sum()}')
print(f'  Invalides (tokens non-int) : {(~rej.valid).sum()}')

print('\nFichiers rejetés par famille :')
print(rej['family'].value_counts())

## 8. Sauvegarde du résumé EDA

In [ ]:
import json

summary = {
    'total_files': int(len(df)),
    'normal_files': int(n_normal),
    'attack_files': int(n_attack),
    'ratio_normal_to_attack': round(ratio, 2),
    'attack_scenarios': int(df[df.label == 1]['scenario'].nunique()),
    'attack_families': df[df.label == 1]['family'].unique().tolist(),
    'length_stats': {
        'normal':  df[df.label == 0]['length'].describe().round(1).to_dict(),
        'attack':  df[df.label == 1]['length'].describe().round(1).to_dict()
    },
    'top_15_syscalls_normal':  [{'syscall': s, 'count': int(c)} for s, c in top_normal[:15]],
    'top_15_syscalls_attack':  [{'syscall': s, 'count': int(c)} for s, c in top_attack[:15]],
    'anomalies': {
        'empty_files': int((df.length == 0).sum()),
        'too_short': int(((df.length > 0) & (df.length < 10)).sum()),
        'invalid_tokens': int((~df.valid).sum()),
        'strange_names': int(df.filename.str.contains('=').sum())
    },
    'cleaning_rules': [
        'length >= 10',
        'all tokens are integers'
    ],
    'files_after_cleaning': kept,
    'files_rejected': rejected
}

with open(OUT / 'eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Résumé sauvegardé dans :', OUT / 'eda_summary.json')
print()
print(json.dumps(summary, indent=2)[:1500] + '...')

## 9. Conclusions

À remplir après exécution :

- **Total chargé:** ?
- **Anomalies trouvées:** ?
- **Règles de nettoyage validées:** ?
- **Signal discriminant confirmé:** poll(168) et clock_gettime(265) dominent les attaques ? ✅ / ❌

**Prochaine étape:** Phase 2 — créer `notebooks/02_modeling.ipynb` pour le pipeline preprocess + train + eval.